# Chapter 7 — Support Vector Machines and Kernel Methods

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Machine Learning  
**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas Support Vector Machine (SVM), konsep margin dan support vectors, kernel linear dan nonlinear, tuning parameter, serta evaluasi model.

## Theory — Support Vector Machine

SVM mencari decision boundary yang memaksimalkan margin antar kelas.
Observasi yang berada paling dekat dengan boundary disebut support vectors.
Parameter `C` mengatur trade-off antara margin yang lebar dan penalti kesalahan
klasifikasi. Scaling fitur penting untuk SVM dan sebaiknya diletakkan di dalam
pipeline agar tidak terjadi data leakage saat cross-validation.

In [ ]:
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score
)

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0, probability=True, random_state=42))
])
linear_svm.fit(X_train, y_train)
pred = linear_svm.predict(X_test)
prob = linear_svm.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=data.target_names))

## Theory — Kernel Methods

Kernel trick memungkinkan SVM membentuk batas keputusan nonlinear tanpa
menghitung transformasi fitur berdimensi tinggi secara eksplisit.

- `linear`: sesuai untuk pola yang dapat dipisahkan secara linear.
- `rbf`: dapat membentuk batas nonlinear; `gamma` mengatur jangkauan pengaruh sampel.
- `poly`: menangkap hubungan polynomial.

Tidak ada kernel yang selalu terbaik; pilih berdasarkan validasi.

In [ ]:
import pandas as pd

X_moon, y_moon = make_moons(n_samples=800, noise=0.25, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moon, y_moon, test_size=0.25, random_state=42, stratify=y_moon
)

results = []
for kernel in ["linear", "rbf", "poly"]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=1.0, gamma="scale"))
    ])
    model.fit(Xm_train, ym_train)
    results.append({
        "kernel": kernel,
        "test_accuracy": accuracy_score(ym_test, model.predict(Xm_test))
    })

pd.DataFrame(results).sort_values("test_accuracy", ascending=False)

## Theory — Hyperparameter Tuning

Parameter utama SVM adalah `C`, `gamma`, dan jenis kernel. GridSearchCV
menggunakan cross-validation pada training set untuk memilih kombinasi
parameter. Test set disimpan untuk evaluasi akhir.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])
param_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.1, 1, 10]},
    {"svm__kernel": ["rbf"], "svm__C": [0.1, 1, 10],
     "svm__gamma": ["scale", 0.01, 0.1, 1]},
    {"svm__kernel": ["poly"], "svm__C": [0.1, 1, 10],
     "svm__degree": [2, 3], "svm__gamma": ["scale"]}
]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1
)
search.fit(X_train, y_train)

best_pred = search.predict(X_test)
print("Best parameters:", search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_test, best_pred), 4))
print(classification_report(y_test, best_pred, target_names=data.target_names))

## Analysis and Conclusion

SVM mengoptimalkan margin pemisah kelas. Kernel linear digunakan untuk pola
linear, sedangkan RBF dan polynomial dapat menangani pola nonlinear. Scaling
dan tuning parameter penting untuk memperoleh generalisasi yang baik.

**Conclusion:** Bandingkan kernel menggunakan cross-validation, dan gunakan
test set hanya untuk evaluasi akhir.